# 🛰️ Siamese-SNN Change Detection — GPU Training

**Agentic Geospatial Compliance System — Model Training Notebook**

This notebook trains the hybrid Siamese-SNN model on the OSCD (Onera Satellite Change Detection) dataset using a **T4 GPU** for ~15-20x speedup over CPU.

### Steps:
1. Install dependencies
2. Upload your OSCD dataset ZIP
3. Train with GPU-optimized hyperparameters
4. Download the trained model

---
**Runtime → Change runtime type → T4 GPU** before running!

## 1. Install Dependencies

In [ ]:
!pip install -q torch torchvision snntorch rasterio pillow scipy tqdm

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## 2. Upload OSCD Dataset

**Option A**: Upload a ZIP of your `data/oscd/` folder from Google Drive  
**Option B**: Upload directly from your machine

The ZIP should contain city folders like:
```
oscd/
├── aguasclaras/
│   ├── imgs_1/
│   ├── imgs_2/
│   └── cm/
├── bercy/
└── ...
```

In [ ]:
import os

# ===== OPTION A: Google Drive (recommended for large files) =====
# Uncomment these lines if your dataset is on Google Drive:

# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/oscd_dataset.zip /content/oscd_dataset.zip
# !unzip -q /content/oscd_dataset.zip -d /content/data/oscd/

# ===== OPTION B: Direct upload =====
# This will open a file picker to upload your ZIP

from google.colab import files
print("Upload your oscd_dataset.zip (the ZIP of data/oscd/ folder):")
uploaded = files.upload()

# Extract
zip_name = list(uploaded.keys())[0]
!mkdir -p /content/data/oscd
!unzip -q "{zip_name}" -d /content/data/oscd/
print("\n✅ Dataset extracted!")

In [ ]:
# Verify the dataset structure
import os
from pathlib import Path

oscd_root = Path("/content/data/oscd")

# Auto-detect if there's a nested folder
# (e.g., ZIP extracted to /content/data/oscd/oscd/aguasclaras)
cities_expected = ["aguasclaras", "bercy", "bordeaux", "paris", "mumbai"]
if not any((oscd_root / c).exists() for c in cities_expected):
    # Check one level deeper
    for sub in oscd_root.iterdir():
        if sub.is_dir() and any((sub / c).exists() for c in cities_expected):
            oscd_root = sub
            print(f"Found dataset at: {oscd_root}")
            break

found = [d.name for d in oscd_root.iterdir() if d.is_dir()]
print(f"Found {len(found)} city folders: {sorted(found)}")

# Quick check first city
for city in found[:1]:
    city_path = oscd_root / city
    has_imgs1 = (city_path / "imgs_1").exists()
    has_imgs2 = (city_path / "imgs_2").exists()
    has_cm = (city_path / "cm").exists()
    print(f"  {city}: imgs_1={'✅' if has_imgs1 else '❌'} imgs_2={'✅' if has_imgs2 else '❌'} cm={'✅' if has_cm else '❌'}")

## 3. Model Architecture (Self-Contained)
All model code is embedded below — no external imports needed.

In [ ]:
"""
============================================================
  ENCODER: Siamese U-Net Encoder with shared weights
============================================================
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple, Optional

try:
    import snntorch as snn
    from snntorch import surrogate, spikegen
    import snntorch.functional as SF
    HAS_SNNTORCH = True
except ImportError:
    HAS_SNNTORCH = False
    print('WARNING: snntorch not found, using fallback mode')


class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        features = self.conv_block(x)
        pooled = self.pool(features)
        return features, pooled


class SiameseEncoder(nn.Module):
    def __init__(self, in_channels=13, encoder_channels=None):
        super().__init__()
        if encoder_channels is None:
            encoder_channels = [64, 128, 256, 512]
        self.encoder_channels = encoder_channels
        channels = [in_channels] + encoder_channels
        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(channels[i], channels[i+1]) for i in range(len(encoder_channels))
        ])
        self.bottleneck = nn.Sequential(
            nn.Conv2d(encoder_channels[-1], encoder_channels[-1]*2, 3, padding=1, bias=False),
            nn.BatchNorm2d(encoder_channels[-1]*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(encoder_channels[-1]*2, encoder_channels[-1]*2, 3, padding=1, bias=False),
            nn.BatchNorm2d(encoder_channels[-1]*2),
            nn.ReLU(inplace=True),
        )

    def encode_single(self, x):
        skips = []
        for block in self.encoder_blocks:
            features, x = block(x)
            skips.append(features)
        bottleneck = self.bottleneck(x)
        return bottleneck, skips

    def forward(self, img_a, img_b):
        bottleneck_a, skips_a = self.encode_single(img_a)
        bottleneck_b, skips_b = self.encode_single(img_b)
        diff_bottleneck = torch.abs(bottleneck_a - bottleneck_b)
        diff_skips = [torch.abs(sa - sb) for sa, sb in zip(skips_a, skips_b)]
        return diff_bottleneck, diff_skips

print('✅ Encoder loaded')

In [ ]:
"""
============================================================
  SNN DECODER: Leaky neurons with surrogate gradients
============================================================
"""

class SNNDecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, beta=0.9):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        concat_channels = out_channels + skip_channels
        self.conv1 = nn.Sequential(
            nn.Conv2d(concat_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        if HAS_SNNTORCH:
            spike_grad = surrogate.fast_sigmoid(slope=25)
            self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=False)
            self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=False)
        else:
            self.lif1 = self.lif2 = None
            self.relu = nn.ReLU(inplace=True)

    def forward(self, x, skip, mem1=None, mem2=None):
        x = self.upsample(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        if HAS_SNNTORCH and self.lif1 is not None:
            if mem1 is None: mem1 = self.lif1.init_leaky()
            spk1, mem1 = self.lif1(x, mem1)
            x = self.conv2(spk1)
            if mem2 is None: mem2 = self.lif2.init_leaky()
            spk2, mem2 = self.lif2(x, mem2)
            return spk2, mem1, mem2
        else:
            x = self.relu(x)
            x = self.conv2(x)
            x = self.relu(x)
            return x, None, None


class SNNDecoder(nn.Module):
    def __init__(self, encoder_channels=None, num_classes=2, beta=0.9, num_steps=20):
        super().__init__()
        if encoder_channels is None:
            encoder_channels = [64, 128, 256, 512]
        self.num_steps = num_steps
        self.num_classes = num_classes
        bottleneck_channels = encoder_channels[-1] * 2
        self.decoder_blocks = nn.ModuleList()
        in_ch = bottleneck_channels
        for i in range(len(encoder_channels)-1, -1, -1):
            skip_ch = encoder_channels[i]
            out_ch = encoder_channels[i]
            self.decoder_blocks.append(SNNDecoderBlock(in_ch, skip_ch, out_ch, beta=beta))
            in_ch = out_ch
        self.output_conv = nn.Conv2d(encoder_channels[0], num_classes, kernel_size=1)

    def forward(self, bottleneck_spikes, skip_diffs):
        T = bottleneck_spikes.shape[0] if bottleneck_spikes.dim() == 5 else self.num_steps
        reversed_skips = list(reversed(skip_diffs))
        spk_recordings, mem_recordings = [], []
        block_mems = [(None, None) for _ in self.decoder_blocks]
        for t in range(T):
            x = bottleneck_spikes[t] if bottleneck_spikes.dim() == 5 else bottleneck_spikes
            for i, (block, skip) in enumerate(zip(self.decoder_blocks, reversed_skips)):
                m1, m2 = block_mems[i]
                x, m1, m2 = block(x, skip, m1, m2)
                block_mems[i] = (m1, m2)
            out = self.output_conv(x)
            spk_recordings.append(out)
            mem_recordings.append(out.clone())
        return spk_recordings, mem_recordings

print('✅ SNN Decoder loaded')

In [ ]:
"""
============================================================
  SIAMESE-SNN: Full model assembly
============================================================
"""

class SiameseSNN(nn.Module):
    def __init__(self, in_channels=13, encoder_channels=None, num_classes=2, beta=0.9, num_steps=20):
        super().__init__()
        if encoder_channels is None:
            encoder_channels = [64, 128, 256, 512]
        self.num_steps = num_steps
        self.num_classes = num_classes
        self.encoder = SiameseEncoder(in_channels=in_channels, encoder_channels=encoder_channels)
        self.decoder = SNNDecoder(encoder_channels=encoder_channels, num_classes=num_classes, beta=beta, num_steps=num_steps)

    def forward(self, img_a, img_b):
        diff_bottleneck, diff_skips = self.encoder(img_a, img_b)
        if HAS_SNNTORCH:
            diff_norm = torch.sigmoid(diff_bottleneck)
            spike_trains = spikegen.rate(diff_norm, num_steps=self.num_steps)
        else:
            spike_trains = diff_bottleneck.unsqueeze(0).repeat(self.num_steps, 1, 1, 1, 1)
        spk_recordings, mem_recordings = self.decoder(spike_trains, diff_skips)
        return spk_recordings, mem_recordings

    def predict(self, img_a, img_b, threshold=0.3):
        self.eval()
        with torch.no_grad():
            spk_recordings, _ = self.forward(img_a, img_b)
            spk_stack = torch.stack(spk_recordings, dim=0)
            firing_rate = spk_stack.mean(dim=0)
            probs = torch.softmax(firing_rate, dim=1)
            change_prob = probs[:, 1]
            prediction = (change_prob > threshold).long()
        return prediction

print('✅ SiameseSNN model loaded')

In [ ]:
"""
============================================================
  LOSS FUNCTIONS: Combined CE-Rate + Weighted-BCE + Dice
============================================================
"""

class CERateLoss(nn.Module):
    def __init__(self):
        super().__init__()
        if HAS_SNNTORCH:
            self._ce_rate = SF.ce_rate_loss()

    def forward(self, spk_recordings, targets):
        if HAS_SNNTORCH:
            spk_stack = torch.stack(spk_recordings, dim=0)
            T, B, C, H, W = spk_stack.shape
            spk_flat = spk_stack.permute(0, 1, 3, 4, 2).reshape(T, B*H*W, C)
            target_flat = targets.reshape(B*H*W).long()
            return self._ce_rate(spk_flat, target_flat).squeeze()
        else:
            spk_stack = torch.stack(spk_recordings, dim=0)
            firing_rate = spk_stack.mean(dim=0)
            return F.cross_entropy(firing_rate, targets.long())


class WeightedBCELoss(nn.Module):
    def __init__(self, change_weight=15.0):
        super().__init__()
        self.change_weight = change_weight

    def forward(self, spk_recordings, targets):
        spk_stack = torch.stack(spk_recordings, dim=0)
        firing_rate = spk_stack.mean(dim=0)
        probs = torch.softmax(firing_rate, dim=1)
        change_prob = probs[:, 1]
        weight = torch.ones_like(targets, dtype=torch.float32)
        weight[targets == 1] = self.change_weight
        loss = F.binary_cross_entropy(
            change_prob.clamp(1e-7, 1-1e-7), targets.float(), weight=weight)
        return loss


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, spk_recordings, targets):
        spk_stack = torch.stack(spk_recordings, dim=0)
        firing_rate = spk_stack.mean(dim=0)
        probs = torch.softmax(firing_rate, dim=1)
        change_prob = probs[:, 1]
        intersection = (change_prob * targets.float()).sum()
        union = change_prob.sum() + targets.float().sum()
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice


class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=0.2, change_weight=15.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce_rate = CERateLoss()
        self.weighted_bce = WeightedBCELoss(change_weight=change_weight)
        self.dice = DiceLoss()

    def forward(self, spk_recordings, targets):
        loss_ce = self.ce_rate(spk_recordings, targets)
        loss_bce = self.weighted_bce(spk_recordings, targets)
        loss_dice = self.dice(spk_recordings, targets)
        total = self.alpha * loss_ce + (1-self.alpha-self.gamma) * loss_bce + self.gamma * loss_dice
        return total

print('✅ Loss functions loaded')

In [ ]:
"""
============================================================
  DATASET: OSCD loader with multi-resolution band handling
============================================================
"""
import numpy as np
from torch.utils.data import Dataset
from pathlib import Path

try:
    import rasterio
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False

try:
    from scipy.ndimage import zoom as scipy_zoom
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

OSCD_TRAIN_CITIES = [
    'aguasclaras', 'bercy', 'bordeaux', 'nantes', 'paris', 'rennes',
    'saclay_e', 'abudhabi', 'cupertino', 'pisa', 'beihai',
    'hongkong', 'beirut', 'mumbai'
]
OSCD_TEST_CITIES = [
    'brasilia', 'montpellier', 'norcia', 'rio', 'saclay_w',
    'valencia', 'dubai', 'lasvegas', 'milano', 'chongqing'
]


def _find_band_file(directory, band_name):
    directory = Path(directory)
    simple = directory / f'{band_name}.tif'
    if simple.exists(): return simple
    matches = list(directory.glob(f'*_{band_name}.tif'))
    if matches: return matches[0]
    matches = list(directory.glob(f'*_{band_name.upper()}.tif'))
    if matches: return matches[0]
    return None


class OSCDDataset(Dataset):
    def __init__(self, root_dir, split='train', patch_size=256, bands=None, augment=True, normalize=True):
        super().__init__()
        self.root_dir = Path(root_dir)
        self.split = split
        self.patch_size = patch_size
        self.bands = bands or ['B02','B03','B04','B08']
        self.augment = augment and split == 'train'
        self.normalize = normalize
        self._ref_dims = {}

        if split == 'train':
            self.cities = [c for c in OSCD_TRAIN_CITIES if (self.root_dir/c).exists()]
        elif split == 'test':
            self.cities = [c for c in OSCD_TEST_CITIES if (self.root_dir/c).exists()]
        else:
            self.cities = [c for c in OSCD_TRAIN_CITIES+OSCD_TEST_CITIES if (self.root_dir/c).exists()]

        self.use_mock = len(self.cities) == 0
        if self.use_mock:
            print('WARNING: No cities found, using mock data!')
            self.cities = ['mock']

        self.patches = self._build_patch_index()
        print(f'OSCD {split}: {len(self.cities)} cities, {len(self.patches)} patches')

    def _get_reference_dims(self, city):
        if city in self._ref_dims: return self._ref_dims[city]
        city_dir = self.root_dir / city / 'imgs_1'
        h, w = 512, 512
        for bname in ['B02','B03','B04','B08']:
            bfile = _find_band_file(city_dir, bname)
            if bfile and HAS_RASTERIO:
                with rasterio.open(str(bfile)) as src:
                    h, w = src.height, src.width
                break
        self._ref_dims[city] = (h, w)
        return h, w

    def _build_patch_index(self):
        patches = []
        if self.use_mock:
            return [{'city':'mock','row':i*self.patch_size,'col':j*self.patch_size} for i in range(10) for j in range(10)]
        for city in self.cities:
            h, w = self._get_reference_dims(city)
            stride = self.patch_size // 2 if self.split == 'train' else self.patch_size
            for row in range(0, h - self.patch_size + 1, stride):
                for col in range(0, w - self.patch_size + 1, stride):
                    patches.append({'city': city, 'row': row, 'col': col})
        return patches

    def _load_bands(self, city, time_dir):
        city_dir = self.root_dir / city / time_dir
        ref_h, ref_w = self._get_reference_dims(city)
        band_arrays = []
        for band_name in self.bands:
            band_file = _find_band_file(city_dir, band_name)
            if band_file and HAS_RASTERIO:
                with rasterio.open(str(band_file)) as src:
                    data = src.read(1).astype(np.float32)
                if data.shape != (ref_h, ref_w):
                    if HAS_SCIPY:
                        data = scipy_zoom(data, (ref_h/data.shape[0], ref_w/data.shape[1]), order=1)
                        data = data[:ref_h, :ref_w]
                        if data.shape[0] < ref_h or data.shape[1] < ref_w:
                            data = np.pad(data, ((0, ref_h-data.shape[0]), (0, ref_w-data.shape[1])))
                    else:
                        ri = np.linspace(0, data.shape[0]-1, ref_h).astype(int)
                        ci = np.linspace(0, data.shape[1]-1, ref_w).astype(int)
                        data = data[np.ix_(ri, ci)]
                band_arrays.append(data)
            else:
                band_arrays.append(np.zeros((ref_h, ref_w), dtype=np.float32))
        return np.stack(band_arrays, axis=0)

    def _load_change_mask(self, city):
        cm_file = self.root_dir / city / 'cm' / 'cm.png'
        ref_h, ref_w = self._get_reference_dims(city)
        if cm_file.exists():
            from PIL import Image
            mask = np.array(Image.open(str(cm_file)))
            if mask.ndim == 3: mask = mask[:,:,0]
            mask = (mask > 128).astype(np.float32)
            if mask.shape != (ref_h, ref_w):
                from PIL import Image as PILImage
                mask_pil = PILImage.fromarray((mask*255).astype(np.uint8))
                mask_pil = mask_pil.resize((ref_w, ref_h), PILImage.NEAREST)
                mask = (np.array(mask_pil) > 128).astype(np.float32)
        else:
            mask = np.zeros((ref_h, ref_w), dtype=np.float32)
        return mask

    def __len__(self): return len(self.patches)

    def __getitem__(self, idx):
        p = self.patches[idx]
        city, row, col = p['city'], p['row'], p['col']

        if self.use_mock:
            C, H = len(self.bands), self.patch_size
            return torch.rand(C,H,H), torch.rand(C,H,H), torch.randint(0,2,(H,H))

        img_before = self._load_bands(city, 'imgs_1')
        img_after = self._load_bands(city, 'imgs_2')
        mask = self._load_change_mask(city)

        r_end = min(row+self.patch_size, img_before.shape[1])
        c_end = min(col+self.patch_size, img_before.shape[2])
        img_before = img_before[:, row:r_end, col:c_end]
        img_after = img_after[:, row:r_end, col:c_end]
        mask = mask[row:r_end, col:c_end]

        _, ph, pw = img_before.shape
        if ph < self.patch_size or pw < self.patch_size:
            pad_h, pad_w = self.patch_size-ph, self.patch_size-pw
            img_before = np.pad(img_before, ((0,0),(0,pad_h),(0,pad_w)))
            img_after = np.pad(img_after, ((0,0),(0,pad_h),(0,pad_w)))
            mask = np.pad(mask, ((0,pad_h),(0,pad_w)))

        if self.normalize:
            for c in range(img_before.shape[0]):
                for img in [img_before, img_after]:
                    bmin, bmax = img[c].min(), img[c].max()
                    if bmax > bmin: img[c] = (img[c]-bmin)/(bmax-bmin)

        if self.augment:
            if np.random.random() > 0.5:
                img_before = np.flip(img_before, 2).copy()
                img_after = np.flip(img_after, 2).copy()
                mask = np.flip(mask, 1).copy()
            if np.random.random() > 0.5:
                img_before = np.flip(img_before, 1).copy()
                img_after = np.flip(img_after, 1).copy()
                mask = np.flip(mask, 0).copy()
            k = np.random.randint(0, 4)
            if k > 0:
                img_before = np.rot90(img_before, k, axes=(1,2)).copy()
                img_after = np.rot90(img_after, k, axes=(1,2)).copy()
                mask = np.rot90(mask, k, axes=(0,1)).copy()

        return (
            torch.from_numpy(img_before.copy()).float(),
            torch.from_numpy(img_after.copy()).float(),
            torch.from_numpy(mask.copy()).long(),
        )

print('✅ Dataset loader ready')

## 4. GPU-Optimized Training

Key differences from CPU training:
- **Batch size**: 8 (vs 2 on CPU)
- **SNN time-steps**: 20 (vs 5-8 on CPU)
- **Full encoder channels**: [64, 128, 256, 512] (vs [32, 64, 128, 256])
- **All 13 bands** or 4-band mode (your choice)
- **Mixed precision (FP16)** for 2x speed on T4

In [ ]:
# ========================================
#  CONFIGURATION — Adjust these as needed
# ========================================

GPU_CONFIG = {
    'epochs': 50,
    'batch_size': 8,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'patch_size': 128,
    'num_steps': 20,         # Full SNN time-steps on GPU!
    'num_bands': 4,          # 4 or 13
    'encoder_channels': [64, 128, 256, 512],  # Full-size encoder
    'gradient_clip': 1.0,
    'change_weight': 15.0,
    'predict_threshold': 0.3,
}

BANDS_4 = ['B02', 'B03', 'B04', 'B08']
BANDS_13 = ['B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B10','B11','B12']

# Choose bands
USE_BANDS = BANDS_4 if GPU_CONFIG['num_bands'] == 4 else BANDS_13

print('GPU Config:')
for k, v in GPU_CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
import time
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Dataset ────────────────────────────────────────────────
# Use the oscd_root detected earlier
train_ds = OSCDDataset(str(oscd_root), split='train', patch_size=GPU_CONFIG['patch_size'],
                       bands=USE_BANDS, augment=True)
val_ds = OSCDDataset(str(oscd_root), split='test', patch_size=GPU_CONFIG['patch_size'],
                     bands=USE_BANDS, augment=False)

# If no test cities have labels, use a subset of train for validation
if len(val_ds) == 0 or val_ds.use_mock:
    print('No test cities found. Splitting train data 80/20.')
    total = len(train_ds)
    train_size = int(0.8 * total)
    val_size = total - train_size
    train_ds, val_ds = torch.utils.data.random_split(train_ds, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=GPU_CONFIG['batch_size'], shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=GPU_CONFIG['batch_size'], shuffle=False,
                        num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} patches ({len(train_loader)} batches)')
print(f'Val:   {len(val_ds)} patches ({len(val_loader)} batches)')

# ── Model ──────────────────────────────────────────────────
model = SiameseSNN(
    in_channels=GPU_CONFIG['num_bands'],
    encoder_channels=GPU_CONFIG['encoder_channels'],
    beta=0.9,
    num_steps=GPU_CONFIG['num_steps'],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} parameters')

# ── Loss / Optimizer ──────────────────────────────────────
loss_fn = CombinedLoss(alpha=0.5, gamma=0.2, change_weight=GPU_CONFIG['change_weight'])
optimizer = AdamW(model.parameters(), lr=GPU_CONFIG['lr'], weight_decay=GPU_CONFIG['weight_decay'])
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
scaler = GradScaler()  # Mixed precision for T4

print('\n✅ Ready to train!')

In [ ]:
# ══════════════════════════════════════════════════════════
#  TRAINING LOOP — GPU optimized with mixed precision
# ══════════════════════════════════════════════════════════

import os
os.makedirs('/content/outputs/models', exist_ok=True)

best_f1 = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_iou': []}
threshold = GPU_CONFIG['predict_threshold']

print('=' * 70)
print(f'  🚀 Training: {GPU_CONFIG["epochs"]} epochs | batch={GPU_CONFIG["batch_size"]} | '
      f'LR={GPU_CONFIG["lr"]} | T={GPU_CONFIG["num_steps"]}')
print('=' * 70)

for epoch in range(1, GPU_CONFIG['epochs'] + 1):
    epoch_start = time.time()

    # ── Train ──
    model.train()
    total_loss = 0.0
    for batch_idx, batch in enumerate(train_loader):
        img_a = batch[0].to(device)
        img_b = batch[1].to(device)
        mask = batch[2].to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass
        with autocast():
            spk_rec, _ = model(img_a, img_b)
            loss = loss_fn(spk_rec, mask)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GPU_CONFIG['gradient_clip'])
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

        if (batch_idx + 1) % 20 == 0:
            print(f'  Epoch {epoch} [{batch_idx+1}/{len(train_loader)}] loss={loss.item():.4f}', end='\r')

    avg_train_loss = total_loss / max(len(train_loader), 1)
    history['train_loss'].append(avg_train_loss)

    # ── Validate ──
    model.eval()
    tp = fp = fn = tn = 0
    val_loss_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            img_a = batch[0].to(device).float()
            img_b = batch[1].to(device).float()
            mask = batch[2].to(device).long()

            with autocast():
                spk_rec, _ = model(img_a, img_b)
                val_loss_total += loss_fn(spk_rec, mask).item()

            pred = model.predict(img_a, img_b, threshold=threshold)
            tp += ((pred == 1) & (mask == 1)).sum().item()
            fp += ((pred == 1) & (mask == 0)).sum().item()
            fn += ((pred == 0) & (mask == 1)).sum().item()
            tn += ((pred == 0) & (mask == 0)).sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)
    avg_val_loss = val_loss_total / max(len(val_loader), 1)

    history['val_loss'].append(avg_val_loss)
    history['val_f1'].append(f1)
    history['val_iou'].append(iou)
    scheduler.step(f1)

    elapsed = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]['lr']

    print(f'Epoch {epoch:3d}/{GPU_CONFIG["epochs"]} | '
          f'Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | '
          f'F1: {f1:.4f} | IoU: {iou:.4f} | P: {precision:.3f} R: {recall:.3f} | '
          f'LR: {current_lr:.2e} | Time: {elapsed:.0f}s')

    # Save best
    if f1 > best_f1:
        best_f1 = f1
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'metrics': {'f1': f1, 'iou': iou, 'precision': precision, 'recall': recall},
            'best_f1': best_f1,
            'config': GPU_CONFIG,
        }
        torch.save(checkpoint, '/content/outputs/models/best_model.pt')
        print(f'  ★ New best model! F1={best_f1:.4f}')

    # Periodic checkpoint
    if epoch % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_f1': best_f1,
            'config': GPU_CONFIG,
        }, f'/content/outputs/models/checkpoint_epoch_{epoch}.pt')
        print(f'  → Checkpoint saved: epoch_{epoch}.pt')

print('\n' + '=' * 70)
print(f'✅ Training complete! Best F1: {best_f1:.4f}')
print(f'Best model → /content/outputs/models/best_model.pt')
print('=' * 70)

## 5. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', color='#2196F3')
axes[0].plot(history['val_loss'], label='Val Loss', color='#F44336')
axes[0].set_title('Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1
axes[1].plot(history['val_f1'], label='Val F1', color='#4CAF50', linewidth=2)
axes[1].axhline(y=best_f1, color='gold', linestyle='--', label=f'Best F1={best_f1:.4f}')
axes[1].set_title('F1 Score', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

# IoU
axes[2].plot(history['val_iou'], label='Val IoU', color='#9C27B0', linewidth=2)
axes[2].set_title('IoU', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Siamese-SNN Training Progress', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/outputs/training_curves.png', dpi=150)
plt.show()

## 6. Download Trained Model

Download the best model to use in your local project.

In [ ]:
# Quick inference test
model.eval()
with torch.no_grad():
    test_a = torch.randn(1, GPU_CONFIG['num_bands'], 128, 128).to(device)
    test_b = torch.randn(1, GPU_CONFIG['num_bands'], 128, 128).to(device)
    pred = model.predict(test_a, test_b)
    print(f'Inference test: input={test_a.shape} → output={pred.shape}')
    print(f'Change pixels detected: {pred.sum().item()} / {pred.numel()}')
    print(f'\nBest F1: {best_f1:.4f}')
    print(f'Model saved at: /content/outputs/models/best_model.pt')

In [ ]:
# Download the best model
from google.colab import files
files.download('/content/outputs/models/best_model.pt')
print('\n📥 Model downloaded! Place it in: outputs/models/best_model.pt')

In [ ]:
# Optional: Also download training curves
files.download('/content/outputs/training_curves.png')

## 7. (Optional) Save to Google Drive

For persistent storage across Colab sessions.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/outputs/models/best_model.pt /content/drive/MyDrive/best_model.pt
# !cp /content/outputs/training_curves.png /content/drive/MyDrive/training_curves.png
# print('✅ Saved to Google Drive!')